# nanoGPT 🤗

A simplified GPT language model trained on text data from [karpathy/climbmix-400b-shuffle](https://huggingface.co/datasets/karpathy/climbmix-400b-shuffle).

The model learns to predict the next token given a sequence of previous tokens.

**Architecture:** Embedding → Multi-Head Attention → MLP (Linear → ReLU → Linear) → Dropout → Linear (vocab_size)

## 1. Load Dataset

Raw text data stored as a plain `.txt` file (29MB, ~10k documents).

In [ ]:
with open('./data/climbmix_sample.txt', 'r') as f:
    text = f.read()

print(f"Total characters: {len(text):,}")
print(f"First 500 chars:\n{text[:500]}")

## 2. Tokenizer

Using `BertTokenizer` (vocab size: 30,522). Created once and saved locally for reuse.

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokenizer.save_pretrained('./mi_tokenizer_bert_local')
print(f"Vocab size: {tokenizer.vocab_size}")

## 3. Tokenize Dataset

Encode the full text into token IDs and save to disk. **Run only once** — afterwards, just load from the file.

In [ ]:
tokenized_dataset = tokenizer.encode(text)
with open('./data/tokenized.txt', 'w') as f:
    f.write(' '.join(map(str, tokenized_dataset)))

In [ ]:
with open('./data/tokenized.txt', 'r') as f:
    data = list(map(int, f.read().split()))

print(f"Total tokens: {len(data):,}")
print(f"First 20: {data[:20]}")

## 4. Custom Dataset

Split the token sequence into fixed-length sentences (`block_size`). For each sentence, generate multiple training examples by slicing at different positions (`min_cut` to `max_cut`):

| X (input, padded) | Y (target) |
|---|---|
| `[0, 0, 0, 0, tok0, tok1, tok2, tok3, tok4]` | `tok5` |
| `[0, 0, 0, tok0, tok1, tok2, tok3, tok4, tok5]` | `tok6` |
| `...` | `...` |
| `[tok0, tok1, tok2, tok3, tok4, tok5, tok6, tok7, tok8]` | `tok9` |

Shorter sequences are left-padded with zeros to keep a uniform input length.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class NanoGPTDataset(Dataset):
    def __init__(self, data, block_size, min_cut, max_cut):
        sentences = [data[i:i+block_size] for i in range(0, len(data), block_size)]
        sentences = [s for s in sentences if len(s) == block_size]

        self.max_len = max_cut - 1
        self.X = []
        self.Y = []
        for s in sentences:
            for cut in range(min_cut, max_cut):
                padded = [0] * (self.max_len - cut) + s[:cut]
                self.X.append(padded)
                self.Y.append(s[cut])

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.long), torch.tensor(self.Y[idx], dtype=torch.long)

## 5. Model

Single-layer Transformer decoder with causal masking. The model predicts the next token from the last position of the output sequence.

```
Embedding(vocab_size, d_model)
       ↓
MultiheadAttention (causal mask)
       ↓
Linear(d_model, 4*d_model) → ReLU → Linear(4*d_model, d_model)
       ↓
Dropout
       ↓
Linear(d_model, vocab_size)
```

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.attention = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4 * d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1), device=x.device)
        att, _ = self.attention(emb, emb, emb, attn_mask=mask, is_causal=True)
        x = self.fc1(att)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.dropout(x)
        x = self.lm_head(x)
        return x[:, -1, :]

## 6. Train and Test Loops

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss_val = loss.item()
            current = batch * dataloader.batch_size + len(X)
            print(f"loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f}\n")

## 7. Setup

Using the first 1M tokens for faster training. 80/20 train/test split.

In [ ]:
block_size = 10
dataset = NanoGPTDataset(data[:1000000], block_size, min_cut=5, max_cut=10)
print(f"Total examples: {len(dataset)}")

train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=512)

vocab_size = tokenizer.vocab_size
model = NanoGPT(vocab_size=vocab_size, d_model=128, nhead=8, dropout=0.1)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

learning_rate = 1e-3
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## 8. Training

In [ ]:
epochs = 5

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")